# 02 - Data preprocessing | Ames Housing 

**Project**

### Import libreries    and read data from __../data/__

In [69]:
from pathlib import Path 
import json, joblib 
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# Support running Jupyter from the repository root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "ames.csv").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "ames.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"


df = pd.read_csv(RAW_PATH)
df.head()

,Unnamed: 0,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### Define the target variable 

In [5]:
TARGET = "SalePrice"
#Verificate if the target is in the dataframe and ensensial conditions for the target variable
assert TARGET in df, f"Expected target {TARGET!r}. Found:{df.columns.tolist()}"
assert df[TARGET].notna().all(), "Target contains nulls: investigate before proceeding."
assert (df[TARGET]>0).all(), "log1p assumes nonnegative target; review unexpected values."
assert df.index.is_unique, "Expected unique index; found duplicates."

#Print the target variable information
print("Raw shape:", df.shape)
df.info()

print("----------------------------------------------")
print("----------------------------------------------")
print("Statistical summary:")
print("Target skew:", round(df[TARGET].skew(), 3))
print("Fraction of rows containing any missing predictor:",
      f"{df.drop(columns=TARGET).isna().any(axis=1).mean():.1%}")
print("Most missing predictors (descriptive only):")
print(df.drop(columns=TARGET).isna().mean().sort_values(ascending=False).head(8).to_string())

Raw shape: (2930, 83)
<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 83 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       2930 non-null   int64  
 1   Order            2930 non-null   int64  
 2   PID              2930 non-null   int64  
 3   MS SubClass      2930 non-null   int64  
 4   MS Zoning        2930 non-null   str    
 5   Lot Frontage     2440 non-null   float64
 6   Lot Area         2930 non-null   int64  
 7   Street           2930 non-null   str    
 8   Alley            198 non-null    str    
 9   Lot Shape        2930 non-null   str    
 10  Land Contour     2930 non-null   str    
 11  Utilities        2930 non-null   str    
 12  Lot Config       2930 non-null   str    
 13  Land Slope       2930 non-null   str    
 14  Neighborhood     2930 non-null   str    
 15  Condition 1      2930 non-null   str    
 16  Condition 2      2930 non-null   str    
 17  Bld

## Data quality 
From the EDA, we expect mixed numerical/categorical features and missing values. Some missing values describe the absence of a feature (for example, no garage). For this baseline, categorical missing values are treated as an explicit Missing category. We do not drop rows or remove outliers without evidence.

In [9]:
if TARGET not in df.columns:
    raise ValueError(f"Missing target column: {TARGET}")
if df[TARGET].isna().any() or (df[TARGET] <= 0).any():
    raise ValueError("SalePrice must be present and strictly positive")

print(f"SalePrice skewness: {df[TARGET].skew():.2f}")
print("Top 10 columns by missing rate:")
display(df.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_rate").style.format({"missing_rate": "{:.1%}"}))

SalePrice skewness: 1.74
Top 10 columns by missing rate:


,missing_rate
Pool QC,99.6%
Misc Feature,96.4%
Alley,93.2%
Fence,80.5%
Mas Vnr Type,60.6%
Fireplace Qu,48.5%
Lot Frontage,16.7%
Garage Cond,5.4%
Garage Finish,5.4%
Garage Yr Blt,5.4%


We found the missing rate of the columns in the dataset. Where the set of Pool QC, Misc Feature, Alley, Fence, Mas Vnr Type and Fireplace QU have  the more than 50% misisng values. We will drop these columns from the dataset.

### Select train/test split

In [40]:
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=SEED)
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

print(f"Train: {len(df_train)} rows, Test: {len(df_test)} rows ")
assert len(df_train) + len(df_test) == len(df), "Train/test split does not match original data"

Train: 2344 rows, Test: 586 rows 


### Select predictions

In [62]:
#this list contains the columns that will be dropped from the dataframe, if they exist in the dataframe
DROP_COLS = [c for c in ["Order", "PID", "Unnamed: 0"] if c in df.columns]

#Split the data into features and target variable for both train and test sets
X_train = df_train.drop(columns=[TARGET, *DROP_COLS]).copy()
X_test = df_test.drop(columns=[TARGET, *DROP_COLS]).copy()
y_train = df_train[TARGET].copy()
y_test = df_test[TARGET].copy()

#Convertir MS SubClass(categorical category) to string if it exists in the dataframe
if "MS SubClass" in X_train.columns:
    X_train["MS SubClass"] = X_train["MS SubClass"].astype(str)
    X_test["MS SubClass"] = X_test["MS SubClass"].astype(str)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

### Train-only feactures check

In [63]:
correlations = X_train[numeric_features].corrwith(y_train).dropna()
display(
    correlations.reindex(correlations.abs().sort_values(ascending=False).index)
    .head(10)
    .rename("pearson_r")
    .to_frame()
)

,pearson_r
Overall Qual,0.795298
Gr Liv Area,0.698315
Garage Cars,0.644304
Garage Area,0.633106
Total Bsmt SF,0.612256
1st Flr SF,0.607433
Year Built,0.545409
Full Bath,0.542053
Year Remod/Add,0.517653
Garage Yr Blt,0.516211


### Build the preprocessing pipeline

In [66]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape: ", X_test_processed.shape)

Processed train shape: (2344, 330)
Processed test shape:  (586, 330)


In [71]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

### Validate the output

In [67]:
from scipy import sparse

def all_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return np.isfinite(values).all()

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert all_finite(X_train_processed) and all_finite(X_test_processed)

print("Checks passed ✓")
print("Final feature count:", len(preprocessor.get_feature_names_out()))

Checks passed ✓
Final feature count: 330


### Save reproducible preprocessing artifacts

In [70]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(PROCESSED_DIR / "train.csv", index=False)
df_test.to_csv(PROCESSED_DIR / "test.csv", index=False)
np.savez(PROCESSED_DIR / "split_indices.npz", train=train_idx, test=test_idx)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Saved:")
print("- data/processed/train.csv")
print("- data/processed/test.csv")
print("- data/processed/split_indices.npz")
print("- models/preprocessor.joblib")

Saved:
- data/processed/train.csv
- data/processed/test.csv
- data/processed/split_indices.npz
- models/preprocessor.joblib
